In [1]:
import json
from pathlib import Path
from imgnet.collections.store import IndexedDatasets
from imgnet.collections.source import ZenodoSource
from imgnet.download import download_collection
from imgtools.nifti.crawl import Crawler
from rich import print

/home/gpudual/bhklab/josh/med-image_index/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
indexed_datasets = IndexedDatasets("indexed_datasets") # This will download the latest version of the indexed_datasets if you dont provide a path
print(indexed_datasets.collections[:10])

12:58:44 [info     ] Indexed datasets path: /home/gpudual/bhklab/josh/med-image_index/notebooks/indexed_datasets [imgnet] call=store.__init__:46


[
    '4D-Lung',
    'ACRIN-6698',
    'ACRIN-Contralateral-Breast-MR',
    'ACRIN-FLT-Breast',
    'ACRIN-NSCLC-FDG-PET',
    'Adrenal-ACC-Ki67-Seg',
    'Advanced-MRI-Breast-Lesions',
    'Anti-PD-1_Lung',
    'B-mode-and-CEUS-Liver',
    'BREAST-DIAGNOSIS'
]

In [3]:
source = ZenodoSource(
    file_type="nifti",
    source="zenodo",
    record_id="17788162",
    post_download=["unzip"],
    description="The Response Evaluation Criteria in Solid Tumors (RECIST 1.1) protocol is the gold standard for assessing treatment response in oncological clinical trials and routine practice. It requires radiologists to review and select appropriate target lesions, and perform precise diameter measurements, making the process labor-intensive and variable. Artificial Intelligence (AI) holds great promise for automating this workflow, but progress is hindered by the lack of public datasets with comprehensive lesion annotations and RECIST-compliant measurements. We address this gap by presenting a dataset of 1,246 manually segmented lesions from 58 CT scans of 22 cancer patients treated at the Clinical Hospital of the University of Chile (HCUCH). All cases were evaluated under RECIST 1.1, with diameter measurements reported for 82 target lesions. This resource supports diverse applications, including validating automated RECIST tools, applying radiomics to study metastatic heterogeneity, benchmarking segmentation algorithms, and advancing foundation models in medical imaging. By including data from a Latin American institution, this dataset also promotes global representation in the development of generalizable medical AI tools."
)

In [4]:
dataset_index_path = (indexed_datasets.imgtools_path / "HCUCH-recist-dataset")
dataset_index_path.mkdir(parents=True, exist_ok=True)
with open(dataset_index_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

In [5]:
print(indexed_datasets.source_config("HCUCH-recist-dataset"))

ZenodoSource(
    file_type=<FileType.NIFTI: 'nifti'>,
    source='zenodo',
    record_id='17788162',
    filenames=None,
    post_download=['unzip'],
    description='The Response Evaluation Criteria in Solid Tumors (RECIST 1.1) protocol is the gold standard for 
assessing treatment response in oncological clinical trials and routine practice. It requires radiologists to 
review and select appropriate target lesions, and perform precise diameter measurements, making the process 
labor-intensive and variable. Artificial Intelligence (AI) holds great promise for automating this workflow, but 
progress is hindered by the lack of public datasets with comprehensive lesion annotations and RECIST-compliant 
measurements. We address this gap by presenting a dataset of 1,246 manually segmented lesions from 58 CT scans of 
22 cancer patients treated at the Clinical Hospital of the University of Chile (HCUCH). All cases were evaluated 
under RECIST 1.1, with diameter measurements reported for 82 target lesions. This resource supports diverse 
applications, including validating automated RECIST tools, applying radiomics to study metastatic heterogeneity, 
benchmarking segmentation algorithms, and advancing foundation models in medical imaging. By including data from a 
Latin American institution, this dataset also promotes global representation in the development of generalizable 
medical AI tools.'
)

In [7]:
temp_data_path = Path("temp_data") / "HCUCH-recist-dataset"
temp_data_path.mkdir(parents=True, exist_ok=True)
download_collection(temp_data_path, source)

15:07:39 [info     ] Downloading all files from Zenodo record 17788162 [imgnet] call=dispatcher._download_zenodo:94
README.md: 100%|██████████| 8.15k/8.15k [00:00<00:00, 755kB/s]
final-formatted.zip: 100%|██████████| 4.66G/4.66G [16:30<00:00, 4.71MB/s]
15:24:11 [info     ] Unpacking final-formatted.zip  [imgnet] call=utils._post_unzip:142


In [6]:
import imgtools.loggers
import logging

imgtools.loggers.get_logger("imgtools").setLevel(logging.INFO)
temp_data_path = Path("temp_data") / "HCUCH-recist-dataset"

crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="final-formatted/images/{BodyPartExamined}/{Split}/images/{PatientID:d}_{SeriesInstanceUID}.nii.gz",
    mask_name_pattern="final-formatted/images/{BodyPartExamined}/{Split}/masks/{PatientID:d}_{SeriesInstanceUID}.nii.gz",
    output_dir=indexed_datasets.imgtools_path,
    n_jobs=10,
    force=True,
    deep=True,
)
crawler.crawl()

12:59:07 [info     ] Starting NIFTI crawl.          [imgtools] nifti_dir=PosixPath('temp_data/HCUCH-recist-dataset') output_dir=PosixPath('indexed_datasets/.imgtools') dataset_name=None call=crawler.crawl:78
         [info     ] Found 116 NIfTI files in /home/gpudual/bhklab/josh/med-image_index/notebooks/temp_data/HCUCH-recist-dataset. [imgtools] call=parse_niftis.parse_nifti_dir:426
         [info     ] Using shared keys: ['BodyPartExamined', 'Split', 'PatientID', 'SeriesInstanceUID'] for reference_id, this will be used to link masks to their referenced scans [imgtools] call=parse_niftis.parse_nifti_dir:447
13:01:22 [info     ] Parsing all NIfTI files took 134.5251 seconds [imgtools] call=timer_utils.__exit__:58
         [info     ] Saved index.                   [imgtools] path=indexed_datasets/.imgtools/HCUCH-recist-dataset/index.csv rows=116 call=parse_niftis.parse_nifti_dir:492


In [22]:
index_df = indexed_datasets.index("HCUCH-recist-dataset")
index_df.head()


,BodyPartExamined,Split,PatientID,SeriesInstanceUID,filepath,file_type,reference_id,class,hash,size,...,mask.bbox.max_coord,mask.feret_diameter,mask.roundness,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan
0,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_3_1.3.12.2.1107.5.1.4.83504.30000...,MedImage,81ef1a4ffb1dc7d6dd92451bb3e70c5e30b7378c,"(512, 512, 173)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021082509324422...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_3_1.3.12.2.1107.5.1.4.83504.30000...,MedImage,0b0f2d0fbb1d47cc16dd4c89cb3474202b01e3ed,"(512, 512, 175)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021102712332815...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_3_1.3.12.2.1107.5.1.4.83504.30000...,MedImage,bd06c947ae6e5249a8489a9ce8f37bd7e8dd8599,"(512, 512, 177)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,abdomen,test,13,1.3.12.2.1107.5.1.4.83504.30000021061014315422...,final-formatted/images/abdomen/test/images/013...,scan,abdomen_test_13_1.3.12.2.1107.5.1.4.83504.3000...,MedImage,5e67e7fe10e0c53b4f105ad9d4a7fc50ad77a47a,"(512, 512, 203)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,abdomen,test,13,1.3.12.2.1107.5.1.4.83504.30000021081905213109...,final-formatted/images/abdomen/test/images/013...,scan,abdomen_test_13_1.3.12.2.1107.5.1.4.83504.3000...,MedImage,3892d6a599c4f524c17779725e045e3a82b8f13d,"(512, 512, 189)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
import pandas as pd

temp_data_path = Path("temp_data") / "HCUCH-recist-dataset"
patients_df = pd.read_csv(temp_data_path / "final-formatted" / "metadata" / "patients.csv")
patients_df.head()

,patient_id,subset,first_study_date,sex,age,diagnosis,histology,health_insurance
0,1,train,20230426,M,70,lung cancer,NaN,public
1,2,train,20220712,F,62,gastric cancer,NaN,uninsured
2,3,test,20210615,F,61,gallbladder cancer,Poorly Differentiated Adenocarcinoma with Sign...,uninsured
3,5,train,20210527,F,59,colon cancer,Moderately Differentiated Adenocarcinoma,uninsured
4,7,train,20210526,M,53,rectal cancer,Well Differentiated Adenocarcinoma,uninsured


In [24]:
import toml

with (temp_data_path / "final-formatted" / "metadata" / "patients_mapping.toml").open("r") as f:
    patients_mapping = toml.load(f)

non_empty_mappings = {k: v for k, v in patients_mapping.items() if v}

non_empty_mappings

{'patient_id': 'PatientID',
 'first_study_date': 'StudyDate',
 'sex': 'PatientSex',
 'age': 'PatientAge'}

In [25]:
patients_df.rename(columns=non_empty_mappings, inplace=True)
patients_df.head()

,PatientID,subset,StudyDate,PatientSex,PatientAge,diagnosis,histology,health_insurance
0,1,train,20230426,M,70,lung cancer,NaN,public
1,2,train,20220712,F,62,gastric cancer,NaN,uninsured
2,3,test,20210615,F,61,gallbladder cancer,Poorly Differentiated Adenocarcinoma with Sign...,uninsured
3,5,train,20210527,F,59,colon cancer,Moderately Differentiated Adenocarcinoma,uninsured
4,7,train,20210526,M,53,rectal cancer,Well Differentiated Adenocarcinoma,uninsured


In [26]:
index_df = pd.merge(index_df, patients_df, on="PatientID", how="left")
index_df.head()


,BodyPartExamined,Split,PatientID,SeriesInstanceUID,filepath,file_type,reference_id,class,hash,size,...,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,subset,StudyDate,PatientSex,PatientAge,diagnosis,histology,health_insurance
0,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_3_1.3.12.2.1107.5.1.4.83504.30000...,MedImage,81ef1a4ffb1dc7d6dd92451bb3e70c5e30b7378c,"(512, 512, 173)",...,NaN,NaN,NaN,test,20210615,F,61,gallbladder cancer,Poorly Differentiated Adenocarcinoma with Sign...,uninsured
1,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021082509324422...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_3_1.3.12.2.1107.5.1.4.83504.30000...,MedImage,0b0f2d0fbb1d47cc16dd4c89cb3474202b01e3ed,"(512, 512, 175)",...,NaN,NaN,NaN,test,20210615,F,61,gallbladder cancer,Poorly Differentiated Adenocarcinoma with Sign...,uninsured
2,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021102712332815...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_3_1.3.12.2.1107.5.1.4.83504.30000...,MedImage,bd06c947ae6e5249a8489a9ce8f37bd7e8dd8599,"(512, 512, 177)",...,NaN,NaN,NaN,test,20210615,F,61,gallbladder cancer,Poorly Differentiated Adenocarcinoma with Sign...,uninsured
3,abdomen,test,13,1.3.12.2.1107.5.1.4.83504.30000021061014315422...,final-formatted/images/abdomen/test/images/013...,scan,abdomen_test_13_1.3.12.2.1107.5.1.4.83504.3000...,MedImage,5e67e7fe10e0c53b4f105ad9d4a7fc50ad77a47a,"(512, 512, 203)",...,NaN,NaN,NaN,test,20210610,M,58,gastric cancer,Papillary Adenocarcinoma,uninsured
4,abdomen,test,13,1.3.12.2.1107.5.1.4.83504.30000021081905213109...,final-formatted/images/abdomen/test/images/013...,scan,abdomen_test_13_1.3.12.2.1107.5.1.4.83504.3000...,MedImage,3892d6a599c4f524c17779725e045e3a82b8f13d,"(512, 512, 189)",...,NaN,NaN,NaN,test,20210610,M,58,gastric cancer,Papillary Adenocarcinoma,uninsured


In [27]:
recist_df = pd.read_csv(temp_data_path / "final-formatted" / "recist_measurements.csv")
recist_df.head()



,patient_id,subset,study_date,study_uuid,uuid,filename,region,final_3d_objects,lesion_label_value,lesion_label_alias,recist_measurement_mm,study_order
0,1,train,20230426,1.3.51.0.1.1.172.19.3.128.3187796.3187735,1.3.12.2.1107.5.1.4.83504.30000023042612315883...,001_1.3.12.2.1107.5.1.4.83504.3000002304261231...,abdomen,1,1,A,40,baseline
1,1,train,20230426,1.3.51.0.1.1.172.19.3.128.3187796.3187735,1.3.12.2.1107.5.1.4.83504.30000023042612315883...,001_1.3.12.2.1107.5.1.4.83504.3000002304261231...,thorax,2,2,B,24,baseline
2,2,train,20220712,1.3.51.0.1.1.172.19.3.128.3051489.3051428,1.3.12.2.1107.5.1.4.83504.30000022071212080050...,002_1.3.12.2.1107.5.1.4.83504.3000002207121208...,abdomen,2,1,A,27,baseline
3,3,test,20210615,1.3.51.0.1.1.172.19.3.128.2857496.2857435,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,003_1.3.12.2.1107.5.1.4.83504.3000002106150914...,abdomen,3,3,A,61,baseline
4,3,test,20210615,1.3.51.0.1.1.172.19.3.128.2857496.2857435,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,003_1.3.12.2.1107.5.1.4.83504.3000002106150914...,abdomen,3,2,B,23,baseline


In [28]:
with (temp_data_path / "final-formatted" / "recist_measurements_mapping.toml").open("r") as f:
    recist_measurements_mapping = toml.load(f)

non_empty_mappings = {k: v for k, v in recist_measurements_mapping.items() if v}

non_empty_mappings

{'patient_id': 'PatientID',
 'study_date': 'StudyDate',
 'study_uuid': 'StudyID',
 'uuid': 'SeriesInstanceUID'}

In [29]:
recist_df.rename(columns=non_empty_mappings, inplace=True)
recist_df.head()

,PatientID,subset,StudyDate,StudyID,SeriesInstanceUID,filename,region,final_3d_objects,lesion_label_value,lesion_label_alias,recist_measurement_mm,study_order
0,1,train,20230426,1.3.51.0.1.1.172.19.3.128.3187796.3187735,1.3.12.2.1107.5.1.4.83504.30000023042612315883...,001_1.3.12.2.1107.5.1.4.83504.3000002304261231...,abdomen,1,1,A,40,baseline
1,1,train,20230426,1.3.51.0.1.1.172.19.3.128.3187796.3187735,1.3.12.2.1107.5.1.4.83504.30000023042612315883...,001_1.3.12.2.1107.5.1.4.83504.3000002304261231...,thorax,2,2,B,24,baseline
2,2,train,20220712,1.3.51.0.1.1.172.19.3.128.3051489.3051428,1.3.12.2.1107.5.1.4.83504.30000022071212080050...,002_1.3.12.2.1107.5.1.4.83504.3000002207121208...,abdomen,2,1,A,27,baseline
3,3,test,20210615,1.3.51.0.1.1.172.19.3.128.2857496.2857435,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,003_1.3.12.2.1107.5.1.4.83504.3000002106150914...,abdomen,3,3,A,61,baseline
4,3,test,20210615,1.3.51.0.1.1.172.19.3.128.2857496.2857435,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,003_1.3.12.2.1107.5.1.4.83504.3000002106150914...,abdomen,3,2,B,23,baseline


In [30]:
recist_df.columns


Index(['PatientID', 'subset', 'StudyDate', 'StudyID', 'SeriesInstanceUID',
       'filename', 'region', 'final_3d_objects', 'lesion_label_value',
       'lesion_label_alias', 'recist_measurement_mm', 'study_order'],
      dtype='object')

In [31]:
# Drop columns that we already have in the index_df
recist_df.drop(columns=["subset", "filename", "SeriesInstanceUID", "region", "StudyDate"], inplace=True)
recist_df.head()

,PatientID,StudyID,final_3d_objects,lesion_label_value,lesion_label_alias,recist_measurement_mm,study_order
0,1,1.3.51.0.1.1.172.19.3.128.3187796.3187735,1,1,A,40,baseline
1,1,1.3.51.0.1.1.172.19.3.128.3187796.3187735,2,2,B,24,baseline
2,2,1.3.51.0.1.1.172.19.3.128.3051489.3051428,2,1,A,27,baseline
3,3,1.3.51.0.1.1.172.19.3.128.2857496.2857435,3,3,A,61,baseline
4,3,1.3.51.0.1.1.172.19.3.128.2857496.2857435,3,2,B,23,baseline


In [32]:
index_df = pd.merge(index_df, recist_df, on="PatientID", how="left")
index_df.head()

,BodyPartExamined,Split,PatientID,SeriesInstanceUID,filepath,file_type,reference_id,class,hash,size,...,PatientAge,diagnosis,histology,health_insurance,StudyID,final_3d_objects,lesion_label_value,lesion_label_alias,recist_measurement_mm,study_order
0,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_3_1.3.12.2.1107.5.1.4.83504.30000...,MedImage,81ef1a4ffb1dc7d6dd92451bb3e70c5e30b7378c,"(512, 512, 173)",...,61,gallbladder cancer,Poorly Differentiated Adenocarcinoma with Sign...,uninsured,1.3.51.0.1.1.172.19.3.128.2857496.2857435,3.0,3.0,A,61.0,baseline
1,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_3_1.3.12.2.1107.5.1.4.83504.30000...,MedImage,81ef1a4ffb1dc7d6dd92451bb3e70c5e30b7378c,"(512, 512, 173)",...,61,gallbladder cancer,Poorly Differentiated Adenocarcinoma with Sign...,uninsured,1.3.51.0.1.1.172.19.3.128.2857496.2857435,3.0,2.0,B,23.0,baseline
2,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_3_1.3.12.2.1107.5.1.4.83504.30000...,MedImage,81ef1a4ffb1dc7d6dd92451bb3e70c5e30b7378c,"(512, 512, 173)",...,61,gallbladder cancer,Poorly Differentiated Adenocarcinoma with Sign...,uninsured,1.3.51.0.1.1.172.19.3.128.2857496.2857435,3.0,1.0,C,17.0,baseline
3,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_3_1.3.12.2.1107.5.1.4.83504.30000...,MedImage,81ef1a4ffb1dc7d6dd92451bb3e70c5e30b7378c,"(512, 512, 173)",...,61,gallbladder cancer,Poorly Differentiated Adenocarcinoma with Sign...,uninsured,1.3.51.0.1.1.172.19.3.128.2890383.2890322,3.0,3.0,A,61.0,follow-up-1
4,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_3_1.3.12.2.1107.5.1.4.83504.30000...,MedImage,81ef1a4ffb1dc7d6dd92451bb3e70c5e30b7378c,"(512, 512, 173)",...,61,gallbladder cancer,Poorly Differentiated Adenocarcinoma with Sign...,uninsured,1.3.51.0.1.1.172.19.3.128.2890383.2890322,3.0,2.0,B,17.0,follow-up-1


In [33]:
index_df.columns

Index(['BodyPartExamined', 'Split', 'PatientID', 'SeriesInstanceUID',
       'filepath', 'file_type', 'reference_id', 'class', 'hash', 'size',
       'ndim', 'nvoxels', 'spacing', 'origin', 'direction', 'min', 'max',
       'sum', 'mean', 'std', 'variance', 'dtype_str', 'dtype_numpy',
       'mask.bbox.size', 'mask.bbox.min_coord', 'mask.bbox.max_coord',
       'mask.feret_diameter', 'mask.roundness', 'mask.flatness',
       'mask.elongation', 'mask.equivalent_spherical_radius',
       'mask.equivalent_spherical_perimeter',
       'mask.equivalent_ellipsoid_diameters', 'mask.volume_count',
       'reference_scan', 'subset', 'StudyDate', 'PatientSex', 'PatientAge',
       'diagnosis', 'histology', 'health_insurance', 'StudyID',
       'final_3d_objects', 'lesion_label_value', 'lesion_label_alias',
       'recist_measurement_mm', 'study_order'],
      dtype='object')

In [35]:
def map_modality(row):
    if row["file_type"] == "scan":
        return "CT"
    elif row["file_type"] == "mask":
        return "SEG"

index_df["Modality"] = index_df.apply(map_modality, axis=1)
index_df


,BodyPartExamined,Split,PatientID,SeriesInstanceUID,filepath,file_type,reference_id,class,hash,size,...,diagnosis,histology,health_insurance,StudyID,final_3d_objects,lesion_label_value,lesion_label_alias,recist_measurement_mm,study_order,Modality
0,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_3_1.3.12.2.1107.5.1.4.83504.30000...,MedImage,81ef1a4ffb1dc7d6dd92451bb3e70c5e30b7378c,"(512, 512, 173)",...,gallbladder cancer,Poorly Differentiated Adenocarcinoma with Sign...,uninsured,1.3.51.0.1.1.172.19.3.128.2857496.2857435,3.0,3.0,A,61.0,baseline,CT
1,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_3_1.3.12.2.1107.5.1.4.83504.30000...,MedImage,81ef1a4ffb1dc7d6dd92451bb3e70c5e30b7378c,"(512, 512, 173)",...,gallbladder cancer,Poorly Differentiated Adenocarcinoma with Sign...,uninsured,1.3.51.0.1.1.172.19.3.128.2857496.2857435,3.0,2.0,B,23.0,baseline,CT
2,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_3_1.3.12.2.1107.5.1.4.83504.30000...,MedImage,81ef1a4ffb1dc7d6dd92451bb3e70c5e30b7378c,"(512, 512, 173)",...,gallbladder cancer,Poorly Differentiated Adenocarcinoma with Sign...,uninsured,1.3.51.0.1.1.172.19.3.128.2857496.2857435,3.0,1.0,C,17.0,baseline,CT
3,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_3_1.3.12.2.1107.5.1.4.83504.30000...,MedImage,81ef1a4ffb1dc7d6dd92451bb3e70c5e30b7378c,"(512, 512, 173)",...,gallbladder cancer,Poorly Differentiated Adenocarcinoma with Sign...,uninsured,1.3.51.0.1.1.172.19.3.128.2890383.2890322,3.0,3.0,A,61.0,follow-up-1,CT
4,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_3_1.3.12.2.1107.5.1.4.83504.30000...,MedImage,81ef1a4ffb1dc7d6dd92451bb3e70c5e30b7378c,"(512, 512, 173)",...,gallbladder cancer,Poorly Differentiated Adenocarcinoma with Sign...,uninsured,1.3.51.0.1.1.172.19.3.128.2890383.2890322,3.0,2.0,B,17.0,follow-up-1,CT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
575,thorax,train,31,1.3.12.2.1107.5.1.4.83504.30000023081013180867...,final-formatted/images/thorax/train/masks/031_...,mask,thorax_train_31_1.3.12.2.1107.5.1.4.83504.3000...,Mask,ae3e4591274bc978da691359e12dfadd5db9fd50,"(512, 512, 315)",...,lung cancer,NaN,public,1.3.51.0.1.1.172.19.3.128.3268322.3268261,3.0,10.0,A,23.0,follow-up-1,SEG
576,thorax,train,31,1.3.12.2.1107.5.1.4.83504.30000023101713284842...,final-formatted/images/thorax/train/masks/031_...,mask,thorax_train_31_1.3.12.2.1107.5.1.4.83504.3000...,Mask,6161cd6a28d7001596c7009991844e204c6c5906,"(512, 512, 295)",...,lung cancer,NaN,public,1.3.51.0.1.1.172.19.3.128.3238264.3238203,7.0,8.0,A,28.0,baseline,SEG
577,thorax,train,31,1.3.12.2.1107.5.1.4.83504.30000023101713284842...,final-formatted/images/thorax/train/masks/031_...,mask,thorax_train_31_1.3.12.2.1107.5.1.4.83504.3000...,Mask,6161cd6a28d7001596c7009991844e204c6c5906,"(512, 512, 295)",...,lung cancer,NaN,public,1.3.51.0.1.1.172.19.3.128.3268322.3268261,3.0,10.0,A,23.0,follow-up-1,SEG
578,thorax,train,37,1.3.12.2.1107.5.1.4.83885.30000023022112155417...,final-formatted/images/thorax/train/masks/037_...,mask,thorax_train_37_1.3.12.2.1107.5.1.4.83885.3000...,Mask,efffce88219b1e31f964ab36bead48905797b5a6,"(512, 512, 361)",...,lung cancer,NaN,uninsured,1.3.51.0.1.1.172.19.3.128.3156833.3156772,3.0,1.0,A,25.0,baseline,SEG


In [36]:
index_df.to_csv(indexed_datasets.imgtools_path / "HCUCH-recist-dataset" / "index.csv", index=False)